# 02 · Rotation Matrices in 3D

### Recap & why now
Project 1 built the $2\times2$ rotation matrix, and Project 3 used it for a drone that
could only tilt one way. Three dimensions needs three of them — one per axis — and
introduces a property that has no 2-D equivalent: **rotations do not commute**.

Roll-then-pitch lands somewhere different from pitch-then-roll. That single fact is
why 3-D orientation is harder than an angle, and it shapes everything that follows.

### Learning objectives
1. Write $R_x$, $R_y$ and $R_z$ and say which axis each leaves untouched.
2. Verify the two properties every rotation matrix has: $R^\top R = I$ and $\det R = 1$.
3. **Compose** rotations by multiplying, and read the order correctly.
4. Demonstrate non-commutativity numerically and measure how far apart the results land.
5. Use $R^\top$ to rotate a vector back from world to body.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection that every figure here needs.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print matrices with 3 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=3, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Orientation toolkit, built up over Notebooks 02-05 ==================

def quat_normalize(q):
    """Force |q| = 1. Integration drifts off the unit sphere; this pulls it back."""
    q = np.asarray(q, float)
    return q/np.linalg.norm(q)

def quat_multiply(a, b):
    """Hamilton product a (x) b: 'do b first, then a', the same reading order as matrices."""
    aw, ax, ay, az = a
    bw, bx, by, bz = b
    return np.array([aw*bw - ax*bx - ay*by - az*bz,     # Scalar part.
                     aw*bx + ax*bw + ay*bz - az*by,     # Vector part, x.
                     aw*by - ax*bz + ay*bw + az*bx,     #              y.
                     aw*bz + ax*by - ay*bx + az*bw])    #              z.

def quat_conjugate(q):
    """Flip the vector part — for a unit quaternion this is the INVERSE rotation."""
    return np.array([q[0], -q[1], -q[2], -q[3]])

def quat_to_rotmat(q):
    """The body-to-world rotation matrix that this quaternion represents."""
    w, x, y, z = quat_normalize(q)
    return np.array([[1-2*(y*y+z*z),   2*(x*y-w*z),   2*(x*z+w*y)],
                     [  2*(x*y+w*z), 1-2*(x*x+z*z),   2*(y*z-w*x)],
                     [  2*(x*z-w*y),   2*(y*z+w*x), 1-2*(x*x+y*y)]])

def euler_to_quat(roll, pitch, yaw):
    """ZYX Euler angles -> quaternion. Used to SET a pose, never to store one."""
    cr, sr = np.cos(roll/2), np.sin(roll/2)
    cp, sp = np.cos(pitch/2), np.sin(pitch/2)
    cy, sy = np.cos(yaw/2), np.sin(yaw/2)
    return np.array([cr*cp*cy + sr*sp*sy, sr*cp*cy - cr*sp*sy,
                     cr*sp*cy + sr*cp*sy, cr*cp*sy - sr*sp*cy])

def quat_to_euler(q):
    """Quaternion -> roll, pitch, yaw. For DISPLAY only — never as simulator state."""
    w, x, y, z = quat_normalize(q)
    return np.array([np.arctan2(2*(w*x + y*z), 1 - 2*(x*x + y*y)),
                     np.arcsin(np.clip(2*(w*y - z*x), -1, 1)),      # clip guards against 1+1e-16.
                     np.arctan2(2*(w*z + x*y), 1 - 2*(y*y + z*z))])

# === The vehicle, and how to draw it =====================================

PARAMS = dict(m=1.0, L=0.25,                       # Mass [kg] and hub-to-rotor distance [m].
              I=np.diag([0.01, 0.01, 0.02]),       # Inertia [kg m^2]; yaw is the heavy axis.
              d=0.016,                             # Drag torque per newton of thrust [m].
              T_min=0.0, T_max=6.0)                # What one motor can produce [N].
g = 9.81                                           # Gravity [m/s^2], along world -z.

ARM = PARAMS["L"]/np.sqrt(2)                       # Each rotor sits ARM along body x AND body y.
MOTOR_POS = np.array([[ ARM, -ARM, 0.0],           # M1 front-right.
                      [ ARM,  ARM, 0.0],           # M2 front-left.
                      [-ARM,  ARM, 0.0],           # M3 rear-left.
                      [-ARM, -ARM, 0.0]])          # M4 rear-right.
SPIN = np.array([-1.0, 1.0, -1.0, 1.0])            # +1 = counter-clockwise seen from above.

def draw_quad(ax, position, q, scale=3.0, thrusts=None):
    """Draw the drone: four arms, four rotors, a nose marker and the thrust arrow."""
    R = quat_to_rotmat(q)                          # Body-to-world, so body points become world points.
    for i, mp in enumerate(MOTOR_POS):
        tip = np.asarray(position, float) + R @ (mp*scale)
        seg = np.array([position, tip])
        ax.plot(seg[:, 0], seg[:, 1], seg[:, 2], color="0.35", lw=2)
        shade = "C3" if i in (0, 1) else "C0"      # Front rotors red, rear blue, so the nose is visible.
        if thrusts is not None:
            load = np.clip(thrusts[i]/PARAMS["T_max"], 0, 1)
            shade = plt.cm.YlOrRd(0.3 + 0.7*load)  # Colour by how hard the motor is working.
        ax.plot([tip[0]], [tip[1]], [tip[2]], "o", ms=6, color=shade)
    ax.quiver(*position, *(R[:, 2]*0.9), color="C1", lw=2.2, arrow_length_ratio=0.25)

def set_3d(ax, xlim, ylim, zlim):
    """Equal-ish 3-D axes with explicit limits, so animations do not jitter."""
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_zlim(*zlim)
    ax.set_box_aspect([xlim[1]-xlim[0], ylim[1]-ylim[0], zlim[1]-zlim[0]])
    ax.set_xlabel("x — East [m]"); ax.set_ylabel("y — North [m]"); ax.set_zlabel("z — Up [m]")

print("vehicle ready: %.1f kg, hover %.2f N total, %.3f N per motor, thrust/weight %.2f" %
      (PARAMS["m"], PARAMS["m"]*g, PARAMS["m"]*g/4, 4*PARAMS["T_max"]/(PARAMS["m"]*g)))

## 1 · Three elementary rotations

$$R_x(\phi) = \begin{bmatrix} 1&0&0 \\ 0&\cos\phi&-\sin\phi \\ 0&\sin\phi&\cos\phi \end{bmatrix},
\quad
R_y(\theta) = \begin{bmatrix} \cos\theta&0&\sin\theta \\ 0&1&0 \\ -\sin\theta&0&\cos\theta \end{bmatrix},
\quad
R_z(\psi) = \begin{bmatrix} \cos\psi&-\sin\psi&0 \\ \sin\psi&\cos\psi&0 \\ 0&0&1 \end{bmatrix}$$

Each one leaves its own axis alone — look for the row and column of ones — and mixes
the other two exactly the way Project 1's $2\times2$ matrix did. A 3-D rotation matrix
is three 2-D rotations wearing a trenchcoat.

In [ ]:
def Rx(phi):
    """Rotation about the body x-axis, through the nose. This is ROLL."""
    c_, s_ = np.cos(phi), np.sin(phi)
    return np.array([[1, 0, 0], [0, c_, -s_], [0, s_, c_]])

def Ry(theta):
    """Rotation about the body y-axis, out the left side. This is PITCH."""
    c_, s_ = np.cos(theta), np.sin(theta)
    return np.array([[c_, 0, s_], [0, 1, 0], [-s_, 0, c_]])

def Rz(psi):
    """Rotation about the body z-axis, through the rotors. This is YAW."""
    c_, s_ = np.cos(psi), np.sin(psi)
    return np.array([[c_, -s_, 0], [s_, c_, 0], [0, 0, 1]])

def quat_from_R(R):
    """Rotation matrix -> quaternion, so we can hand any R to draw_quad."""
    tr = np.trace(R)
    if tr > 0:
        s_ = np.sqrt(tr + 1.0)*2                   # The safe branch when the trace is positive.
        q = np.array([0.25*s_, (R[2,1]-R[1,2])/s_, (R[0,2]-R[2,0])/s_, (R[1,0]-R[0,1])/s_])
    elif R[0,0] > R[1,1] and R[0,0] > R[2,2]:
        s_ = np.sqrt(1.0 + R[0,0] - R[1,1] - R[2,2])*2
        q = np.array([(R[2,1]-R[1,2])/s_, 0.25*s_, (R[0,1]+R[1,0])/s_, (R[0,2]+R[2,0])/s_])
    elif R[1,1] > R[2,2]:
        s_ = np.sqrt(1.0 + R[1,1] - R[0,0] - R[2,2])*2
        q = np.array([(R[0,2]-R[2,0])/s_, (R[0,1]+R[1,0])/s_, 0.25*s_, (R[1,2]+R[2,1])/s_])
    else:
        s_ = np.sqrt(1.0 + R[2,2] - R[0,0] - R[1,1])*2
        q = np.array([(R[1,0]-R[0,1])/s_, (R[0,2]+R[2,0])/s_, (R[1,2]+R[2,1])/s_, 0.25*s_])
    return quat_normalize(q)                       # Four branches, so we never divide by a small number.

R_test = Rz(0.9) @ Ry(-0.4) @ Rx(0.25)             # An arbitrary composed rotation.
print("R^T R - I, largest entry: %.2e   (should be zero)" % np.abs(R_test.T @ R_test - np.eye(3)).max())
print("det R = %.12f                     (should be exactly +1)" % np.linalg.det(R_test))
print("\nTogether those two say R rotates without stretching or reflecting — and they mean")
print("R inverse is just R transpose, which is a transpose instead of a matrix inversion.")

## 2 · Seeing each one act

The clearest way to understand a rotation matrix is to watch what it does to the
drone's own axes. Below, each elementary rotation is applied to a level quadcopter, and
the grey arrows show where the world axes stayed.

In [ ]:
fig = plt.figure(figsize=(13, 3.8))
for i, (name, R) in enumerate([("roll  $R_x(30°)$", Rx(np.deg2rad(30))),
                               ("pitch $R_y(30°)$", Ry(np.deg2rad(30))),
                               ("yaw   $R_z(30°)$", Rz(np.deg2rad(30)))]):
    ax = fig.add_subplot(1, 3, i+1, projection="3d")
    for j in range(3):
        ax.quiver(0, 0, 0, *(np.eye(3)[:, j]*0.8), color="0.8", lw=1.2, arrow_length_ratio=0.15)
    for j, col in enumerate(["C3", "C2", "C0"]):
        ax.quiver(0, 0, 0, *(R[:, j]*0.95), color=col, lw=2.2, arrow_length_ratio=0.15)
    set_3d(ax, (-1.1, 1.1), (-1.1, 1.1), (-0.6, 1.1))
    ax.set_title(name, fontsize=10); ax.view_init(elev=20, azim=-62)
plt.tight_layout(); plt.show()

z_body = np.array([0.0, 0.0, 1.0])                 # The thrust axis, in body coordinates.
print("thrust axis after 20° roll :", np.round(Rx(np.deg2rad(20)) @ z_body, 3), " -> leans toward -y")
print("thrust axis after 20° pitch:", np.round(Ry(np.deg2rad(20)) @ z_body, 3), " -> leans toward +x")
print("thrust axis after 20° yaw  :", np.round(Rz(np.deg2rad(20)) @ z_body, 3), " -> unchanged ✔")

## 3 · Order matters, and by a lot

In two dimensions you can rotate by 30° then 45°, or 45° then 30°, and end up in the
same place. In three dimensions you cannot. Matrix multiplication does not commute, and
neither do the rotations it represents.

$$R_x R_z \ne R_z R_x$$

The consequence is that "roll 90° and pitch 90°" is not an instruction until you say
which first. Every convention in robotics exists to remove that ambiguity.

In [ ]:
v = np.array([1.0, 0.0, 0.0])                      # A vector along the nose.
first_x = Rz(np.pi/2) @ (Rx(np.pi/2) @ v)          # Roll first, then yaw.
first_z = Rx(np.pi/2) @ (Rz(np.pi/2) @ v)          # Yaw first, then roll.

print("roll then yaw:", np.round(first_x, 4))
print("yaw then roll:", np.round(first_z, 4))
angle = np.degrees(np.arccos(np.clip(first_x @ first_z, -1, 1)))
print("the two results are %.1f° apart — not a rounding difference, a different answer." % angle)

fig = plt.figure(figsize=(9.5, 4.0))
for i, (label, R) in enumerate([("roll 90° then yaw 90°", Rz(np.pi/2) @ Rx(np.pi/2)),
                                ("yaw 90° then roll 90°", Rx(np.pi/2) @ Rz(np.pi/2))]):
    ax = fig.add_subplot(1, 2, i+1, projection="3d")
    draw_quad(ax, [0, 0, 0], quat_from_R(R), scale=2.2)
    set_3d(ax, (-1.2, 1.2), (-1.2, 1.2), (-1.0, 1.2))
    ax.set_title(label, fontsize=10); ax.view_init(elev=20, azim=-62)
plt.tight_layout(); plt.show()

## 4 · Going backwards

If $R$ takes a body vector into the world, then $R^\top$ brings a world vector into the
body. That is the operation a drone performs constantly: gravity is known in the world,
and the flight controller needs it in body coordinates to work out which way "down" is
relative to itself.

In [ ]:
q_pose = euler_to_quat(np.deg2rad(20), np.deg2rad(-10), np.deg2rad(45))
R_pose = quat_to_rotmat(q_pose)                    # Body -> world for this attitude.

gravity_world = np.array([0.0, 0.0, -g])           # What the world knows.
gravity_body = R_pose.T @ gravity_world            # What an onboard accelerometer would feel.
print("gravity in the world:", np.round(gravity_world, 3))
print("gravity in the body :", np.round(gravity_body, 3))
print("magnitude preserved : %.6f vs %.6f ✔" % (np.linalg.norm(gravity_world), np.linalg.norm(gravity_body)))

back = R_pose @ gravity_body                       # And back again, to check.
print("\nround trip error: %.2e" % np.abs(back - gravity_world).max())
print("\nThat body-frame gravity vector is precisely what a levelling algorithm uses: if the")
print("drone were level it would read [0, 0, -9.81], and any tilt shows up in the first two")
print("entries. Notebook 04 of Project 4 estimated exactly this kind of quantity.")

## 🧪 Try it yourself

**E1.** Rotations do not commute, yet in Section 3 the two orders gave results only a
certain angle apart. Would that gap grow or shrink for smaller angles? Reason first.

**E2.** Write `rotate_sequence(angles, axes)` applying a list of rotations in order, and
use it to show that two small rotations *almost* commute.

In [ ]:
# --- Solution E1 ---
print("E1: it shrinks, and quadratically. The difference between the two orders is governed by")
print("    the COMMUTATOR, which for small angles is of order the product of the two angles —")
print("    so halving both angles quarters the gap. That is why small-angle attitude control")
print("    can get away with treating rotations as if they added, and why large manoeuvres")
print("    cannot. E2 measures the shrinkage.")

# --- Solution E2 ---
def rotate_sequence(angles_deg, axes):
    """Apply rotations in the order given: the first entry happens FIRST."""
    R = np.eye(3)
    for ang, ax_ in zip(angles_deg, axes):
        step = {"x": Rx, "y": Ry, "z": Rz}[ax_](np.deg2rad(ang))
        R = step @ R                               # Left-multiply: each new rotation wraps the last.
    return R

print("\nE2:  angle    gap between the two orders")
for deg in (90, 45, 20, 10, 5, 1):
    a = rotate_sequence([deg, deg], ["x", "z"])    # Roll then yaw.
    b = rotate_sequence([deg, deg], ["z", "x"])    # Yaw then roll.
    v0 = np.array([1.0, 0.0, 0.0])
    gap = np.degrees(np.arccos(np.clip((a @ v0) @ (b @ v0), -1, 1)))
    print("    %4d° %20.3f°" % (deg, gap))
print("    Halving the angle quarters the gap, exactly as predicted. At 1° the two orders differ")
print("    by a hundredth of a degree — which is why linearised attitude controllers work near")
print("    hover and stop working during aerobatics.")

## 🚁 Mini-project: the non-commuting drone

Animate both orders side by side — roll-then-yaw against yaw-then-roll — starting from
the same level pose. They begin together, move through different intermediate
attitudes, and finish somewhere visibly different.

In [ ]:
n = 60
def pose_sequence(order):
    """Interpolate through two 90° rotations applied in the given order."""
    out = []
    for k in range(n):                             # First rotation, sweeping in.
        a = 90*k/(n-1)
        out.append(rotate_sequence([a, 0], order))
    for k in range(n):                             # Second rotation, sweeping in.
        b = 90*k/(n-1)
        out.append(rotate_sequence([90, b], order))
    return out

seq_a = pose_sequence(["x", "z"])
seq_b = pose_sequence(["z", "x"])

fig = plt.figure(figsize=(9.5, 4.6))

def frame(k):
    fig.clf()
    for i, (seq, label) in enumerate([(seq_a, "roll first, then yaw"), (seq_b, "yaw first, then roll")]):
        ax = fig.add_subplot(1, 2, i+1, projection="3d")
        draw_quad(ax, [0, 0, 0], quat_from_R(seq[k]), scale=2.2)
        set_3d(ax, (-1.2, 1.2), (-1.2, 1.2), (-1.0, 1.2))
        ax.set_title(label, fontsize=10); ax.view_init(elev=20, azim=-62)
    return []

anim = animation.FuncAnimation(fig, frame, frames=2*n, interval=55, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

> **🤖 Robotics connection.** Non-commutativity is why a robot's orientation cannot be
> stored as three independent numbers that you add up. It is the reason attitude
> estimators integrate a matrix or a quaternion rather than three angles, and the reason
> a gyroscope's output must be integrated with care rather than summed. Every
> orientation representation in robotics is an answer to the question this notebook
> raised.

**Where next.** Three angles are still the most human way to *talk* about orientation.
Notebook 03 defines roll, pitch and yaw properly — and then shows the hole in them.